In [ ]:
!pip install datasets spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from datasets import load_dataset
from collections import Counter, defaultdict

# Load the datasets and the spacy engine
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split="train")
nlp = spacy.load("en_core_web_sm")

START = "<START>"

unigram_counts = Counter()
bigram_counts = defaultdict(Counter) # bigram_counts[prev][curr]
context_totals = Counter() # Total appearences of prev

# Only non-empty lines are saved as text
texts = [line["text"].strip() for line in dataset if line["text"].strip()]

# Batched processing with acceleration:
# - n_process=-1 uses all available CPU cores
# - disable=["parser", "ner"] turns off heavy components we don't need for lemmatization
for doc in nlp.pipe(texts, batch_size=256, n_process=-1, disable=["parser", "ner"]):
    lemmas = [token.lemma_.lower() for token in doc if token.is_alpha]

    if not lemmas:
        continue

    tokens = [START] + lemmas

    for i, word in enumerate(tokens):
        if word != START:
            unigram_counts[word] += 1
        if i > 0:
            prev = tokens[i - 1]
            bigram_counts[prev][word] += 1
            context_totals[prev] += 1

print(f"Vocabulary size (unique lemmas): {len(unigram_counts)}")
print(f"Total tokens: {sum(unigram_counts.values())}")
print(f"Total unique bigrams: {sum(len(v) for v in bigram_counts.values())}")
print(f"Count of 'the': {unigram_counts['the']}")
print(f"Count of 'cat': {unigram_counts['cat']}")
print(f"Count of bigram('in', 'the'): {bigram_counts['in']['the']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Vocabulary size (unique lemmas): 54450
Total tokens: 1665841
Total unique bigrams: 594331
Count of 'the': 130770
Count of 'cat': 91
Count of bigram('in', 'the'): 12754


In [ ]:
import math

N_uni = sum(unigram_counts.values())

def p_uni(word):
    """Unigram MLE = C(word) / N_uni"""
    return unigram_counts[word] / N_uni

def p_bi(curr, prev):
    """Bigram MLE = C(prev, curr)/ C(prev)"""
    if context_totals.get(prev,0) == 0:
        return 0.0
    return bigram_counts[prev].get(curr, 0) / context_totals[prev]


def sentence_tokens(s: str) -> list[str]:
    """Preprocessing to make the sentence into a list of tokens"""
    doc = nlp(s)
    return [START] + [token.lemma_.lower() for token in doc if token.is_alpha]

In [ ]:
# --Task 2--
prompt_doc = nlp("I have a house in")
prompt_lemmas = [token.lemma_.lower() for token in prompt_doc if token.is_alpha]

print(prompt_lemmas)
last = prompt_lemmas[-1]

# Find the argmax of w for p(w | last )
# Notice that C(last) is constant, therefore we look for argmax w for numerator

best_word, best_count = bigram_counts[last].most_common(1)[0]
best_prob = best_count / context_totals[last]

print(f"Most likely continuation of '{last}': {best_word}")
print(f"  count = {best_count}")
print(f"  P({best_word} | {last}) = {best_prob:.6f}")


['i', 'have', 'a', 'house', 'in']
Most likely continuation of 'in': the
  count = 12754
  P(the | in) = 0.285919


In [ ]:
SENT_1 = "Brad Pitt was born in Oklahoma"
SENT_2 = "The actor was born in USA"

def log_prob_bigram(s: str) -> float:
    """Log P(s) under bigram MLE. Returns -inf if any bigram has count 0."""
    token_list = sentence_tokens(s)
    total_lp = 0.0
    for i in range(1, len(token_list)):
        p = p_bi(token_list[i], token_list[i - 1])
        if p == 0.0:
            return float("-inf")
        total_lp += math.log(p)
    return total_lp

def joint_perplexity(sentences, log_prob_func) -> float:
    """PP = exp(-l), where l = (1/M) * sum of log P(s_i) over the test set."""
    total_log_p = 0.0
    M = 0
    for s in sentences:
        log_p = log_prob_func(s)
        if log_p == float("-inf"):
            return float("inf")
        total_log_p += log_p
        M += len(sentence_tokens(s)) - 1
    ell = total_log_p / M
    return math.exp(-ell)

def print_bigram_info(s: str):
    token_list = sentence_tokens(s)
    print(f"Tokens : {token_list}")
    for i in range(1, len(token_list)):
        prev, curr = token_list[i - 1], token_list[i]
        c_bi = bigram_counts[prev].get(curr, 0)
        c_prev = context_totals.get(prev, 0)
        p = p_bi(curr, prev)
        marker = "  <-- ZERO" if p == 0.0 else ""
        print(f"    C({prev!r}, {curr!r}) = {c_bi:>5}"
              f" / C({prev!r}) = {c_prev:>6}  ->  p = {p:.4e}{marker}")


print("-- Task 3 --")
for s in (SENT_1, SENT_2):
    print(f"Sentence: {s!r}")
    print_bigram_info(s)
    lp = log_prob_bigram(s)
    p = math.exp(lp) if lp != float("-inf") else 0.0
    print(f"  log P(s) = {lp}")
    print(f"  P(s)     = {p}\n")

pp = joint_perplexity([SENT_1, SENT_2], log_prob_bigram)
print(f"Joint perplexity (M = 12):  {pp}")

-- Task 3 --
Sentence: 'Brad Pitt was born in Oklahoma'
Tokens : ['<START>', 'brad', 'pitt', 'be', 'bear', 'in', 'oklahoma']
    C('<START>', 'brad') =     0 / C('<START>') =  23633  ->  p = 0.0000e+00  <-- ZERO
    C('brad', 'pitt') =     5 / C('brad') =     12  ->  p = 4.1667e-01
    C('pitt', 'be') =     1 / C('pitt') =     18  ->  p = 5.5556e-02
    C('be', 'bear') =   187 / C('be') =  54328  ->  p = 3.4421e-03
    C('bear', 'in') =   107 / C('bear') =    547  ->  p = 1.9561e-01
    C('in', 'oklahoma') =     1 / C('in') =  44607  ->  p = 2.2418e-05
  log P(s) = -inf
  P(s)     = 0.0

Sentence: 'The actor was born in USA'
Tokens : ['<START>', 'the', 'actor', 'be', 'bear', 'in', 'usa']
    C('<START>', 'the') =  3506 / C('<START>') =  23633  ->  p = 1.4835e-01
    C('the', 'actor') =    41 / C('the') = 130719  ->  p = 3.1365e-04
    C('actor', 'be') =    12 / C('actor') =    275  ->  p = 4.3636e-02
    C('be', 'bear') =   187 / C('be') =  54328  ->  p = 3.4421e-03
    C('bear', 'in')

In [ ]:
# --Task 4--
def p_interp(curr, prev, lam_bi=2/3, lam_uni=1/3):
    """Linear interpolation of bigram and unigram MLEs."""
    return lam_bi * p_bi(curr, prev) + lam_uni * p_uni(curr)

def log_prob_interp(s: str) -> float:
    """Log P(s) under the interpolated model."""
    token_list = sentence_tokens(s)
    total_lp = 0.0
    for i in range(1, len(token_list)):
        p = p_interp(token_list[i], token_list[i - 1])
        if p == 0.0:
            return float("-inf")
        total_lp += math.log(p)
    return total_lp

def print_interp_info(s: str):
    """For each position, show how bigram vs unigram contribute to p_interp."""
    token_list = sentence_tokens(s)
    print(f"Tokens : {token_list}")
    for i in range(1, len(token_list)):
        prev, curr = token_list[i - 1], token_list[i]
        pb = p_bi(curr, prev)
        pu = p_uni(curr)
        pi = 2/3 * pb + 1/3 * pu
        bi_share  = (2/3 * pb) / pi * 100 if pi > 0 else 0.0
        uni_share = (1/3 * pu) / pi * 100 if pi > 0 else 0.0
        print(f"    p_bi({curr!r} | {prev!r}) = {pb:.4e}   p_uni({curr!r}) = {pu:.4e}")
        print(f"      -> p_interp = {pi:.4e}   (bi: {bi_share:5.1f}%, uni: {uni_share:5.1f}%)")

print("-- Task 4 --")
for s in (SENT_1, SENT_2):
    print(f"Sentence: {s!r}")
    print_interp_info(s)
    lp = log_prob_interp(s)
    p = math.exp(lp) if lp != float("-inf") else 0.0
    print(f"  log P_interp(s) = {lp}")
    print(f"  P_interp(s)     = {p}\n")

pp = joint_perplexity([SENT_1, SENT_2], log_prob_interp)
print(f"Joint perplexity (M = 12):  {pp}")



-- Task 4 --
Sentence: 'Brad Pitt was born in Oklahoma'
Tokens : ['<START>', 'brad', 'pitt', 'be', 'bear', 'in', 'oklahoma']
    p_bi('brad' | '<START>') = 0.0000e+00   p_uni('brad') = 7.2036e-06
      -> p_interp = 2.4012e-06   (bi:   0.0%, uni: 100.0%)
    p_bi('pitt' | 'brad') = 4.1667e-01   p_uni('pitt') = 1.0805e-05
      -> p_interp = 2.7778e-01   (bi: 100.0%, uni:   0.0%)
    p_bi('be' | 'pitt') = 5.5556e-02   p_uni('be') = 3.2640e-02
      -> p_interp = 4.7917e-02   (bi:  77.3%, uni:  22.7%)
    p_bi('bear' | 'be') = 3.4421e-03   p_uni('bear') = 3.2956e-04
      -> p_interp = 2.4046e-03   (bi:  95.4%, uni:   4.6%)
    p_bi('in' | 'bear') = 1.9561e-01   p_uni('in') = 2.7028e-02
      -> p_interp = 1.3942e-01   (bi:  93.5%, uni:   6.5%)
    p_bi('oklahoma' | 'in') = 2.2418e-05   p_uni('oklahoma') = 7.2036e-06
      -> p_interp = 1.7347e-05   (bi:  86.2%, uni:  13.8%)
  log P_interp(s) = -36.221540771517354
  P_interp(s)     = 1.858594796385011e-16

Sentence: 'The actor was born i